# Init package

In [1]:
%cd /home/FullMouth/code
import os
from pathlib import Path
import shutil
from glob import glob
import json
import math

import torch
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
device = "cuda" if torch.cuda.is_available() else "CPU"
device

/home/FullMouth/code


'cuda'

In [2]:
from fullmouth_util import (
    FM_label, load_json, write_json, INSTRUCT_PROMPT, SENT, ENTITY_LS, ENTITY_DICT
)

from function_util import (
    get_entity_data, get_msg_ls_checkInstruction, get_msg_ls_resultsInstruction, replace_msg_ls_to_user
)

fm_label = FM_label()

In [4]:
r=16
module_lv = 'light'
# module_lv = 'medium'

In [5]:
no_sys_prompt = False
MODEL_NAME =f'Qwen2.5-7B-Instruct'
# MODEL_NAME =f'Qwen2.5-14B-Instruct' 
# MODEL_NAME = 'Mistral-7B-Instruct-v0.3'
# MODEL_NAME = 'Mistral-Small-Instruct-2409'
# MODEL_NAME = 'OLMo-2-1124-13B-DPO'
# MODEL_NAME = 'OLMo-2-1124-7B-DPO'
# MODEL_NAME = "gemma-2-9b-it"; no_sys_prompt = True


model_type = '_revisedTrain_withDesc_withEx_withErrFeed'
model_name = f'{MODEL_NAME}{model_type}'

do_revise = False
version = f'4_n20'
num_of_instructions = 3
validation_threshold = 0.9

data_dir = os.path.join( r'/home/FullMouth/data', f'instruct_{version}', model_name)
train_data_fp = os.path.join(data_dir, 'SFT_training_data.json')
val_data_fp = os.path.join(data_dir, 'SFT_val_data.json')

training_pos_neg_rate = 3
train_val_rate = 0.9

model_root = r'/data_sys/lang_model'

model_dir = os.path.join(model_root, MODEL_NAME)
print(data_dir)

post_fix = f'{num_of_instructions}instructs{str(validation_threshold).replace(".", "")}'
selected_prompt_fp = os.path.join(data_dir, f'selected_prompt_{post_fix}.json')
instruct_dict_selected = load_json(selected_prompt_fp)

/home/FullMouth/data/instruct_4_n20/Qwen2.5-7B-Instruct_revisedTrain_withDesc_withEx_withErrFeed


In [7]:
# from datasets import load_dataset, concatenate_datasets
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.model_selection import train_test_split
import random

In [9]:

def collect_data(data_ls: list, key_entity: str, instruct_ls: list, version: str, isPositive: bool, no_sys_prompt=no_sys_prompt):
    collect_ls = []
    for e_dict in data_ls:
        one_collect_ls = []
        entity_ls = e_dict[ENTITY_DICT].get(key_entity, []) if isPositive else []
        target_sent = e_dict[SENT]
        gold_output_dict = {SENT: target_sent, ENTITY_LS: entity_ls}

        for instruct_prompt in instruct_ls:
            one_check_msg_ls = get_msg_ls_checkInstruction(instruct_prompt, target_sent, key_entity)
            one_check_msg_ls.append( {"role": "assistant", "content": 'Yes' if isPositive else 'No'} )
            if no_sys_prompt: one_check_msg_ls = replace_msg_ls_to_user(one_check_msg_ls)

            one_result_msg_ls = get_msg_ls_resultsInstruction(instruct_prompt, target_sent, add_stop_token=True)
            one_result_msg_ls.append( {"role": "assistant", "content": json.dumps(gold_output_dict, ensure_ascii=False)} )
            if no_sys_prompt: one_result_msg_ls = replace_msg_ls_to_user(one_result_msg_ls)

            one_collect_ls.append(one_check_msg_ls)
            one_collect_ls.append(one_result_msg_ls)
        
        collect_ls.append(one_collect_ls)
    # if isPositive:
    #     print(f'Collected {len(collect_ls)} positive samples for entity {key_entity}')
    # else:
    #     print(f'Collected {len(collect_ls)} negative samples for entity {key_entity}')
    return collect_ls

def replace_msg_ls_to_user(msg_ls: list[dict]) -> list[dict]:
    """
    Some models require user/assistant only
    """
    for i, msg in enumerate(msg_ls):
        if msg['role'] == 'system':
            msg_ls[i]['role'] = 'user'
    
    # Combine consecutive messages with the same role into one message
    combined_msg_ls = []
    for msg in msg_ls:
        if not combined_msg_ls:
            combined_msg_ls.append(msg)
        else:
            last_msg = combined_msg_ls[-1]
            if msg['role'] == last_msg['role']:
                # Combine the content of the messages
                combined_msg_ls[-1]['content'] += '\n' + msg['content']
            else:
                combined_msg_ls.append(msg)
    
    return combined_msg_ls

# Load dataset

In [10]:
re_do = False

POS = 'pos'
NEG = 'neg'

if not re_do and os.path.exists(train_data_fp) and os.path.exists(val_data_fp):
    train_dataset = load_json(train_data_fp)
    print(f'Loaded training dataset from {train_data_fp}, length: {len(train_dataset)}')
    val_datasets = load_json(val_data_fp)
    print(f'Loaded validation dataset from {val_data_fp}, length: {len(val_datasets)}')

else: 
    data_dir = os.path.join(r'/home/FullMouth/data/annotation_v4', 'training_data_swap_revised')
    json_fp_ls = glob(os.path.join(data_dir, '*.json'))
    print(len(json_fp_ls))
    data_ls = []
    for json_fp in json_fp_ls: data_ls.extend(load_json(json_fp))

    collect_dict = {}
    for key_entity in fm_label.gold_entity_ls:
        entity_data_ls, no_entity_data_ls = get_entity_data(data_ls, key_entity)
        # instruct_prompt = instruct_dict[key_entity][INSTRUCT_PROMPT]
        instruct_ls = [instruct_dict_selected[key_entity][idx][INSTRUCT_PROMPT] for idx in instruct_dict_selected[key_entity]]
        
        collect_dict[key_entity] = {POS: collect_data(entity_data_ls, key_entity, instruct_ls, version, isPositive=True),
                                    NEG: collect_data(no_entity_data_ls, key_entity, instruct_ls, version, isPositive=False)}

    train_ls = []
    val_dict = {POS: [], NEG: []}
    for key_entity in fm_label.gold_entity_ls:
        pos_collect_ls = collect_dict[key_entity][POS]
        neg_collect_ls = collect_dict[key_entity][NEG]

        real_pos_neg_rate = len(neg_collect_ls) / len(pos_collect_ls)
        
        train_pos_collect_ls, val_pos_collect_ls = train_test_split(pos_collect_ls, test_size=1-train_val_rate, random_state=42)

        train_neg_collect_ls = random.sample(neg_collect_ls, len(train_pos_collect_ls)* training_pos_neg_rate)

        val_neg_collect_ls = random.sample(neg_collect_ls, int(len(val_pos_collect_ls)* real_pos_neg_rate))

        one_train_ls = train_pos_collect_ls + train_neg_collect_ls
        one_val_ls = val_pos_collect_ls + val_neg_collect_ls
        train_ls.extend( [l for ls in one_train_ls for l in ls])
        val_dict[POS].extend(val_pos_collect_ls)
        val_dict[NEG].extend(val_neg_collect_ls)
        # print(f'Entity: {key_entity}, Trainig samples: {len(one_train_ls)}, Validation samples: {len(one_val_ls)}')
        # print(real_pos_neg_rate, len(pos_collect_ls), len(neg_collect_ls))

        # break

    print(len(train_ls), len(val_dict[POS]), len(val_dict[NEG]))

    # Convert train_ls (list of message dicts) into a HuggingFace Dataset
    train_dataset = Dataset.from_dict({"messages": train_ls})
    
    # Convert val_dict into HuggingFace Datasets for POS and NEG
    val_pos_flat = [l for ls in val_dict[POS] for l in ls]
    val_neg_flat = [l for ls in val_dict[NEG] for l in ls]

    minor_val_neg_flat = val_neg_flat[:3000]  # Balance the negative samples to match positive samples

    val_datasets = {
        POS: Dataset.from_dict({"messages": val_pos_flat}),
        NEG: Dataset.from_dict({"messages": minor_val_neg_flat})
    }

    # Save the datasets to disk
    write_json(train_dataset, train_data_fp)
    write_json(val_datasets, val_data_fp,)
    del train_ls
    del val_dict

    import gc
    gc.collect()
    print("Memory free after saving datasets.")

print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset (pos) size: {len(val_datasets[POS])}")
print(f"Validation dataset (neg) size: {len(val_datasets[NEG])}")

Loaded training dataset from /home/FullMouth/data/instruct_4_n20/Qwen2.5-7B-Instruct_revisedTrain_withDesc_withEx_withErrFeed/SFT_training_data.pkl, length: 15000
Loaded validation dataset from /home/FullMouth/data/instruct_4_n20/Qwen2.5-7B-Instruct_revisedTrain_withDesc_withEx_withErrFeed/SFT_val_data.pkl, length: 2
Train dataset size: 15000
Validation dataset (pos) size: 484
Validation dataset (neg) size: 3000


In [11]:
train_dataset['messages'][0]

[{'content': 'You are an expert in dentistry and clinical natural language processing.\n\nTask:\nDetermine whether the specified target entity is present in the target sentence, following the provided entity instruction.\n\nInstruction:\nExtract Age mentions from the given clinical notes. Age mentions are numerical expressions indicating a patient\'s age, such as \'56 y/o\', \'25 years old\', or \'1-year-old\'. Follow these guidelines for inclusion and exclusion:\n\n### Inclusion Rules:\n1. **Exact Numerical Expressions**: Include any numerical expression followed by \'years old\', \'y/o\', or \'yo\'.\n2. **Abbreviations**: Include \'y/o\' and \'yo\' as abbreviations for \'years old\'.\n3. **Partial Mentions**: Include partial mentions like \'70 y\' or \'15 yo\' if they are clear indicators of age.\n4. **Age with Gender**: Include ages specified with gender like \'62 year old\', \'70 y/o female\', or \'15 yo male\'.\n5. **Numerical Ages with Units**: Include ages specified with other u

# Calculat step

In [12]:
use_flash_attention = True
per_device_batch_size = 2  # reduced from 4 to avoid OOM
gradient_accumulation_steps = 8  # doubled to keep global_batch_size = 16
num_devices = torch.cuda.device_count() if torch.cuda.is_available() else 1  # set >1 if using DDP / multi-GPU

global_batch_size = per_device_batch_size * gradient_accumulation_steps * num_devices
steps_per_epoch = math.ceil(len(train_dataset) / global_batch_size)

save_eval_steps = max(1, steps_per_epoch // 8)
logging_steps = max(1, steps_per_epoch // 16)

steps_per_epoch, save_eval_steps, logging_steps

(938, 117, 58)

# Load models and config

In [13]:
tokenizer = AutoTokenizer.from_pretrained(model_dir, legacy=False)
tokenizer.padding_side = "right"

In [14]:
# BitsAndBytesConfig int-4 config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, 
    bnb_4bit_use_double_quant=True, 
    bnb_4bit_quant_type="nf4", 
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    model_dir,
    quantization_config=bnb_config,
    use_cache=False,
    low_cpu_mem_usage=True,
    attn_implementation="flash_attention_2",
    device_map="auto",
)
# model.config.pretraining_tp = 1 # for llama

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [15]:
print(tokenizer.eos_token, tokenizer.eos_token_id, model.config.eos_token_id)
print(tokenizer.pad_token, tokenizer.pad_token_id, model.config.pad_token_id)
print(tokenizer.bos_token, tokenizer.bos_token_id, model.config.bos_token_id)

if getattr(model, "generation_config", None) is not None:
    if tokenizer.pad_token_id is not None:
        model.generation_config.pad_token_id = tokenizer.pad_token_id
    if tokenizer.eos_token_id is not None:
        model.generation_config.eos_token_id = tokenizer.eos_token_id
    if tokenizer.bos_token_id is not None:
        model.generation_config.bos_token_id = tokenizer.bos_token_id


print(tokenizer.eos_token, tokenizer.eos_token_id, model.config.eos_token_id)
print(tokenizer.pad_token, tokenizer.pad_token_id, model.config.pad_token_id)
print(tokenizer.bos_token, tokenizer.bos_token_id, model.config.bos_token_id)

<|im_end|> 151645 151645
<|endoftext|> 151643 None
None None 151643
<|im_end|> 151645 151645
<|endoftext|> 151643 None
None None 151643


In [16]:
if tokenizer.pad_token is None:
    print("Setting PAD token to EOS token")
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

    model.config.pad_token_id = tokenizer.pad_token_id

In [17]:
print(f"Current Attention Implementation: {model.config._attn_implementation}")

Current Attention Implementation: flash_attention_2


# LoRa config

In [18]:
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from peft.tuners.lora import LoraLayer

if module_lv == 'light':
    model_ls = ["q_proj", "v_proj"]
elif module_lv == 'medium':
    model_ls = ["q_proj", "k_proj", "v_proj", "o_proj"]
elif module_lv == 'full':
    model_ls = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]


peft_config = LoraConfig(
        r=r, lora_alpha=int(2*r),
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM", 
        target_modules = model_ls,
)

# prepare model for training
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False 

In [19]:
model_output_dir = os.path.join(model_root, version, model_name)
if os.path.exists(model_output_dir): shutil.rmtree(model_output_dir); print('delete old model dir')
if not os.path.exists(model_output_dir):
    Path(model_output_dir).mkdir(parents=True, exist_ok=True)
print(model_output_dir)

delete old model dir
/data_sys/lang_model/4_n20/Qwen2.5-7B-Instruct_revisedTrain_withDesc_withEx_withErrFeed


In [20]:
# Add this cell before setting max_seq_length
def analyze_sequence_lengths(dataset, tokenizer, sample_size=10000):
    lengths = []
    sample_indices = range(min(sample_size, len(dataset)))
    
    for idx in sample_indices:
        # Tokenize the text field from your dataset
        # Adjust the field name based on your data structure
        text = dataset[idx]['text'] if 'text' in dataset[idx] else str(dataset[idx])
        tokens = tokenizer(text, truncation=False)
        lengths.append(len(tokens['input_ids']))
    
    lengths = sorted(lengths)
    return {
        'mean': np.mean(lengths),
        'median': np.median(lengths),
        'p95': np.percentile(lengths, 95),
        'p99': np.percentile(lengths, 99),
        'max': max(lengths)
    }

# import numpy as np
# stats = analyze_sequence_lengths(train_dataset, tokenizer)
# print("Sequence length statistics:")
# for k, v in stats.items():
#     print(f"  {k}: {v:.0f}")

# # Set max_seq_length to p95 or p99 to avoid excessive padding
# max_seq_length = int(stats['p95'])  # or stats['p99']
# print(f"\nRecommended max_seq_length: {max_seq_length}")

In [21]:
from trl import SFTConfig, SFTTrainer
# Initialize SFTConfig with training parameters
max_seq_length = 1600 # max sequence length for model and packing of the dataset

args = SFTConfig(
    output_dir=model_output_dir,
    num_train_epochs=2,

    per_device_train_batch_size=per_device_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    gradient_checkpointing=True,

    optim="paged_adamw_32bit",
    lr_scheduler_type="cosine", #"constant",
    learning_rate=2e-4,
    max_grad_norm=1.0, #0.3,
    warmup_ratio=0.1,  #0.03,
    
    bf16=True,
    fp16=False,
    tf32=True,
    
    logging_dir=os.path.join(model_output_dir, "logs"),
    logging_steps=logging_steps,
    
    max_length=max_seq_length,  # Correct parameter for SFT packing/padding
    packing=True,

    disable_tqdm=False,
    eval_on_start=True,  # evaluate before training to get baseline
    eval_strategy="steps",  # or "epoch"
    eval_steps=save_eval_steps,
    save_strategy="steps", #"epoch",
    save_steps=save_eval_steps,
    load_best_model_at_end=True,
    metric_for_best_model="eval_pos_loss",  # Use positive eval loss for best model selection
    eval_accumulation_steps=4,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [22]:
model = get_peft_model(model, peft_config)

# Upcast layer for flash attnetion
if use_flash_attention:
    # from utils.llama_patch import upcast_layer_for_flash_attention
    torch_dtype = torch.bfloat16 if args.bf16 else torch.float16 if args.fp16 else torch.float32
    # model = upcast_layer_for_flash_attention(model, torch_dtype)
    for name, module in model.named_modules():
        if isinstance(module, LoraLayer) or "norm" in name.lower():
            module.to(torch_dtype)
        if ("lm_head" in name or "embed_tokens" in name) and hasattr(module, "weight"):
            module.to(torch_dtype)

We now have every building block we need to create our `SFTTrainer` to start then training our model.

In [23]:
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_datasets,
    args=args,
)

Tokenizing train dataset:   0%|          | 0/15000 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/15000 [00:00<?, ? examples/s]

Tokenizing pos dataset:   0%|          | 0/484 [00:00<?, ? examples/s]

Packing pos dataset:   0%|          | 0/484 [00:00<?, ? examples/s]

Tokenizing neg dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Packing neg dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

In [24]:
# Recalculate steps based on actual packed dataloader length
actual_steps_per_epoch = math.ceil(len(trainer.get_train_dataloader()) / gradient_accumulation_steps)
save_eval_steps = max(1, actual_steps_per_epoch // 8)
logging_steps = max(1, actual_steps_per_epoch // 16)

trainer.args.eval_steps = save_eval_steps
trainer.args.save_steps = save_eval_steps
trainer.args.logging_steps = logging_steps

print(f"Packed steps/epoch: {actual_steps_per_epoch}, eval/save every {save_eval_steps} steps, log every {logging_steps} steps")

Packed steps/epoch: 712, eval/save every 89 steps, log every 44 steps


# Train model

In [25]:
import sys
sys.path.append(r'/home/tg_bot')
from tg_bot_send import send_msg_https
# send_msg_https(f'SFT training start {model_output_dir}!')

In [26]:
try:
    trainer.train() 
    send_msg_https(f'SFT training done for {model_output_dir}!')
except Exception as e:
    send_msg_https(f'SFT training error {model_output_dir}: {str(e)}')
    raise e

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss,Pos Loss,Entropy,Num Tokens,Mean Token Accuracy,Neg Loss
0,No log,No log,1.939966,1.100651,0.000000,0.606276,1.986071
89,1.303031,No log,0.928144,0.982567,1601148.000000,0.748895,0.971691
178,0.324745,No log,0.222274,0.270844,3183448.000000,0.947415,0.209509
267,0.087099,No log,0.094757,0.094767,4762670.000000,0.986416,0.071733
356,0.065097,No log,0.078391,0.071315,6367185.000000,0.987884,0.061172
445,0.062973,No log,0.074604,0.064682,7963265.000000,0.987876,0.057686
534,0.057966,No log,0.071337,0.060826,9570500.000000,0.987926,0.057219
623,0.057185,No log,0.069718,0.060799,11179612.000000,0.988120,0.056773
712,0.058183,No log,0.069042,0.058807,12774742.000000,0.988142,0.055745
801,0.053551,No log,0.068878,0.057539,14366334.000000,0.988220,0.055872


KeyboardInterrupt: 

# Find best checkpoint

In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [28]:
logs = trainer.state.log_history
logs_per_epoch = logs[-1]['step'] /logs[-1]['epoch']

print(logs_per_epoch, steps_per_epoch)

train_losses = []
eval_pos_losses = []
eval_neg_losses = []
train_steps = []
eval_pos_steps = []
eval_neg_steps = []
mean_token_accuracies = []
mean_token_accuracy_steps = []

for log in logs[:]:
    steps = log['step']#/logs_per_epoch
    if 'loss' in log:
        train_steps.append(steps)
        train_losses.append(log['loss'])
    if 'eval_pos_loss' in log:
        eval_pos_losses.append(log['eval_pos_loss'])
        eval_pos_steps.append(steps)
    if 'eval_neg_loss' in log:
        eval_neg_losses.append(log['eval_neg_loss'])
        eval_neg_steps.append(steps)
    if 'mean_token_accuracy' in log:
        mean_token_accuracies.append(log['mean_token_accuracy'])
        mean_token_accuracy_steps.append(steps)

print(f"Train logs: {len(train_losses)}, Eval pos: {len(eval_pos_losses)}, Eval neg: {len(eval_neg_losses)}")

711.8804766269478 938
Train logs: 31, Eval pos: 16, Eval neg: 16


In [29]:
plt.figure(figsize=(10, 6))
plt.plot(train_steps, train_losses, label='Training Loss', color='blue', alpha=0.7)
if eval_pos_losses: 
    plt.plot(eval_pos_steps, eval_pos_losses, label='Eval Loss (Positive)', color='green', marker='o')
if eval_neg_losses: 
    plt.plot(eval_neg_steps, eval_neg_losses, label='Eval Loss (Negative)', color='red', marker='x')
if mean_token_accuracies:
    plt.plot(mean_token_accuracy_steps, mean_token_accuracies, label='Mean Token Accuracy', color='orange', marker='s')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training and Evaluation Loss Curves (Pos vs Neg)')
plt.legend()
plt.grid(True)
plt.show()

# Save the model

In [30]:
step_epoch_dict = {l['step']: float(l['epoch']) for l in logs[:]}
for fn in os.listdir(model_output_dir):
    if fn.startswith('checkpoint-'):
        check_num = int(fn.replace('checkpoint-', ''))
        print(f'{fn:<15s} - {step_epoch_dict.get(check_num, 0):.1f}')

checkpoint-89   - 0.1
checkpoint-178  - 0.3
checkpoint-267  - 0.4
checkpoint-356  - 0.5
checkpoint-445  - 0.6
checkpoint-534  - 0.8
checkpoint-623  - 0.9
checkpoint-712  - 1.0
checkpoint-801  - 1.1
checkpoint-890  - 1.3
checkpoint-979  - 1.4
checkpoint-1068 - 1.5
checkpoint-1157 - 1.6
checkpoint-1246 - 1.8
checkpoint-1335 - 1.9


In [32]:
step_num = 267
ep = '04'
load_model_dir = os.path.join(model_output_dir, f'checkpoint-{step_num}')
generated_model_settings = f'ep{ep}r{r}_{module_lv}'
output_model_name = f'{os.path.basename(model_output_dir)}_{generated_model_settings}'
save_dir = os.path.join('/data_sys/lang_model', version, output_model_name)
print(load_model_dir)
print(save_dir)

/data_sys/lang_model/4_n20/Qwen2.5-7B-Instruct_revisedTrain_withDesc_withEx_withErrFeed/checkpoint-267
/data_sys/lang_model/4_n20/Qwen2.5-7B-Instruct_revisedTrain_withDesc_withEx_withErrFeed_ep04r16_light


In [33]:
if os.path.exists(save_dir): shutil.rmtree(save_dir); print('delete old model dir')

In [34]:
from peft import AutoPeftModelForCausalLM

In [35]:
model = AutoPeftModelForCausalLM.from_pretrained(
    load_model_dir,
    low_cpu_mem_usage=True,
) 
tokenizer = AutoTokenizer.from_pretrained(load_model_dir, legacy=False)
tokenizer.padding_side = "right"

# Merge LoRA and base model
merged_model = model.merge_and_unload()

merged_model.save_pretrained(save_dir, safe_serialization=True)
tokenizer.save_pretrained(save_dir)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/data_sys/lang_model/4_n20/Qwen2.5-7B-Instruct_revisedTrain_withDesc_withEx_withErrFeed_ep04r16_light/tokenizer_config.json',
 '/data_sys/lang_model/4_n20/Qwen2.5-7B-Instruct_revisedTrain_withDesc_withEx_withErrFeed_ep04r16_light/chat_template.jinja',
 '/data_sys/lang_model/4_n20/Qwen2.5-7B-Instruct_revisedTrain_withDesc_withEx_withErrFeed_ep04r16_light/tokenizer.json')

In [36]:
for fn in os.listdir(model_output_dir):
    if fn.startswith('checkpoint-'):
        shutil.rmtree(os.path.join(model_output_dir, fn))